# 🚀 Data Engineering Pipeline Tutorial - Part 2
## MinIO Data Lake & Data Cleansing

### What You'll Learn:
1. MinIO as S3-compatible local storage
2. Medallion Architecture (Bronze/Silver/Gold)
3. Data cleansing techniques
4. Lambda functions for validation

---
## ☁️ Part 4: MinIO Data Lake

### Why MinIO?
| Feature | Benefit |
|---------|--------|
| S3-compatible | Same API as AWS S3 |
| Free | No cloud costs |
| Local | Fast, works offline |
| Docker | Easy setup |

### Medallion Architecture
```
BRONZE          →        SILVER         →        GOLD
Raw data                 Cleaned                 Business-ready
Multiple formats         Parquet only            Transformed
As-is from source        Validated               Aggregated
```

In [ ]:
import boto3
import pandas as pd
import io
from datetime import datetime

# MinIO endpoint - S3-compatible API
MINIO_ENDPOINT = "http://tina-minio:9000"
BUCKET = "data-lake"

class S3Handler:
    """
    Handle S3 operations for data lake (MinIO).
    
    MinIO uses the same boto3 API as AWS S3!
    Only difference: we specify endpoint_url
    """
    
    def __init__(self, bucket: str, endpoint_url: str = MINIO_ENDPOINT):
        self.bucket = bucket
        # endpoint_url tells boto3 to use MinIO instead of AWS
        self.s3 = boto3.client('s3', endpoint_url=endpoint_url)
    
    def _generate_path(self, zone: str, source: str, file_format: str) -> str:
        """
        Generate S3 path with DATE PARTITIONING.
        
        Output: bronze/customers/year=2026/month=01/day=19/file.parquet
        
        Why partition by date?
        - Query only relevant data
        - Easy retention policies
        - Standard data lake pattern
        """
        now = datetime.now()
        return f"{zone}/{source}/year={now.year}/month={now.month:02d}/day={now.day:02d}/{source}_{now.strftime('%H%M%S')}.{file_format}"
    
    def upload_to_bronze(self, df: pd.DataFrame, source_name: str, 
                         file_format: str = "parquet") -> str:
        """
        Upload raw data to Bronze zone.
        
        Why Parquet?
        - Columnar: Fast for analytics
        - Compressed: ~75% smaller than CSV
        - Schema embedded: Self-describing
        """
        s3_path = self._generate_path("bronze", source_name, file_format)
        
        # io.BytesIO = in-memory file (no disk needed)
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False)
        buffer.seek(0)
        
        self.s3.upload_fileobj(buffer, self.bucket, s3_path)
        print(f"✅ Uploaded: s3://{self.bucket}/{s3_path}")
        return s3_path

# Example usage
# s3 = S3Handler("data-lake")
# s3.upload_to_bronze(df, "customers")

### 💡 S3 Path Best Practices

```
s3://data-lake/bronze/customers/year=2026/month=01/day=19/file.parquet
     └─bucket─┘ └zone┘ └source─┘ └────partitions─────┘ └───file───┘
```

**Hive-style partitioning** (`year=`, `month=`, `day=`) enables:
- Automatic partition pruning in queries
- Works with Spark, Athena, Presto

---
## 🧹 Part 5: Data Cleansing (Bronze → Silver)

**Silver zone = trusted data.** Clean once, use everywhere.

### Common Data Quality Issues:
1. Duplicates
2. Null values
3. Inconsistent formats
4. Wrong data types

In [ ]:
import pandas as pd
from typing import Dict, List

class DataCleaner:
    """Data cleansing utilities for Silver zone."""
    
    @staticmethod
    def remove_duplicates(df: pd.DataFrame, subset: List[str] = None) -> pd.DataFrame:
        """Remove duplicate rows."""
        before = len(df)
        df = df.drop_duplicates(subset=subset)
        removed = before - len(df)
        if removed > 0:
            print(f"🗑️ Removed {removed} duplicates")
        return df
    
    @staticmethod
    def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
        """Lowercase, replace spaces with underscores."""
        df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
        return df
    
    @staticmethod
    def handle_nulls(df: pd.DataFrame, strategy: Dict[str, str]) -> pd.DataFrame:
        """
        Handle nulls per column.
        
        Strategies: 'drop', 'mean', 'median', 'zero', 'empty'
        Example: {'price': 'mean', 'name': 'empty'}
        """
        for col, method in strategy.items():
            if col not in df.columns:
                continue
            if method == 'drop':
                df = df.dropna(subset=[col])
            elif method == 'mean':
                df[col] = df[col].fillna(df[col].mean())
            elif method == 'zero':
                df[col] = df[col].fillna(0)
            elif method == 'empty':
                df[col] = df[col].fillna('')
        return df

In [ ]:
# DEMO: Data Cleansing

messy_data = pd.DataFrame({
    'Customer Name': ['  Alice  ', 'Bob', 'Alice', None],
    'Email': ['alice@email.com', 'bob@email.com', 'alice@email.com', 'eve@email.com'],
    'Amount': [100.0, -50.0, 100.0, 200.0]
})

print("BEFORE:")
print(messy_data)

cleaner = DataCleaner()
df = cleaner.standardize_columns(messy_data.copy())
df = cleaner.remove_duplicates(df, subset=['email'])
df = cleaner.handle_nulls(df, {'customer_name': 'drop'})

print("\nAFTER:")
print(df)

---
## 🎯 Part 6: Lambda Functions for Validation

**Lambda** = anonymous one-line function

```python
# Regular function
def is_positive(x):
    return x > 0

# Same as lambda
is_positive = lambda x: x > 0
```

In [ ]:
# Common validation lambdas for data engineering

validations = {
    'is_positive': lambda x: x > 0,
    'has_email': lambda x: '@' in str(x),
    'not_empty': lambda x: len(str(x).strip()) > 0,
    'in_range': lambda x: 0 <= x <= 100,
}

# Test them
print("Testing validations:")
print(f"  is_positive(100): {validations['is_positive'](100)}")
print(f"  is_positive(-5): {validations['is_positive'](-5)}")
print(f"  has_email('test@email.com'): {validations['has_email']('test@email.com')}")
print(f"  has_email('invalid'): {validations['has_email']('invalid')}")

---
## 🎯 Part 2 Summary

| Concept | Usage |
|---------|-------|
| `endpoint_url` | Connect boto3 to MinIO instead of AWS |
| `io.BytesIO` | In-memory file buffer |
| Hive partitioning | `year=2026/month=01/day=19/` |
| `@staticmethod` | Methods that don't need `self` |
| Lambda | Inline validation rules |

### MinIO Console
View your data at: http://localhost:9001
- Username: `minioadmin`
- Password: `minioadmin`